# log-back — faded example 2: Fill the multiply step in z = log(x*y)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-back`. Running the beacon reports progress on the `Backprop: log_back` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: log_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`log-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "log-back"
DD_SUBTOPIC = "Backprop: log_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For `z = log(x*y)` with loss `z.sum()`, back through log gives `dL/d(xy) = 1/(xy)`, then multiply_back0 gives `dL/dx = dL/d(xy) * y = 1/x`. The cancellation only appears after composing the two back fns.

## Faded exercise 2

Complete `compose_log_mul_back(x, y)` which returns `(dL/dx, dL/dy)` for `z = log(x*y)` summed. The log step is given; fill in the multiply backward for the x-parent.

**Fill in:** the multiply_back0 contribution dL/d(xy) * y for the x parent

In [ ]:
import torch as t

def log_back(grad_out, out, x):
    return grad_out / x

def compose_log_mul_back(x, y):
    xy = x * y
    z = t.log(xy)
    dL_dz = t.ones_like(z)
    dL_dxy = log_back(dL_dz, z, xy)
    dL_dx = dL_dxy * y
    dL_dy = dL_dxy * x
    return dL_dx, dL_dy


def _test():
    t.manual_seed(12)
    x = t.rand(5) + 0.4
    y = t.rand(5) + 0.4
    dx, dy = compose_log_mul_back(x, y)
    # independent closed form: dL/dx = 1/x, dL/dy = 1/y
    assert t.allclose(dx, 1.0 / x, atol=1e-6)
    assert t.allclose(dy, 1.0 / y, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def log_back(grad_out, out, x):
    return grad_out / x

def compose_log_mul_back(x, y):
    xy = x * y
    z = t.log(xy)
    dL_dz = t.ones_like(z)
    dL_dxy = log_back(dL_dz, z, xy)
    dL_dx = dL_dxy * y
    dL_dy = dL_dxy * x
    return dL_dx, dL_dy
```
</details>